# Gamil RAG project

### Step 1: Import library

In [19]:
import os
import ollama
from openai import OpenAI
import json
from dotenv import load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
import gradio as gr
from gmail.gmail_function import login_gmail, get_emails_list
from tqdm import tqdm
from IPython.display import display, Markdown

### Step 2: setting variables and environment

In [3]:
load_dotenv(override = True)
ollama_api_key = os.getenv("OLLAMA_API_KEY")
hf_token = os.getenv("HF_TOKEN")
credential_path = r'.credentials/credentials.json' # create credentials from Gmail API
token_path = r'.credentials/token.json' # Token will be created the first time of login to be used for subsequent login
login(hf_token, add_to_git_credential=True)

# # setup ollama
# llm_client = OpenAI(base_url = "http://localhost:11434/v1/", api_key = ollama_api_key)
# llm_model = "gemma3:1b"

# setup lms
llm_client = OpenAI(base_url = "http://localhost:1234/v1/", api_key = "lms")
llm_model = "google/gemma-3-4b"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Step 3: Check if gmail credentials exist

In [4]:
if not os.path.exists(credential_path):
    print(f"Error: Credentials file '{credential_path}' does not exist.")
else:
    print("Credential file exists!")

Credential file exists!


### Step 4: Login gmail with credentials or token

In [5]:
service = login_gmail(credential_path, token_path)

### Step 5: Retrieve Gmail for the past 7 days

In [7]:
emails = get_emails_list(service, days=7)

In [8]:
print(len(emails))

100


In [11]:
if len(emails)>30:
    emails = emails[:30]

### Step 6: Create a summary for every gmail

In [20]:
system_prompt = f"""
You will be given the title and body of an email, help to create a summary for it.
"""

for email in tqdm(emails):
    title = email["title"]
    body = email["body"]
    user_prompt = f""" 
    Please help to provide summary of the email based on title and body. Keep it clear and concise. 
    
    Here is the title of the email:
    f{title}

    Here is the body of the email:
    f{body}
    """
    messages = [{"role":"system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]

    response = llm_client.chat.completions.create(model = llm_model, messages = messages, max_tokens = 2000)

    email["summary"] = response.choices[0].message.content


100%|██████████| 30/30 [03:48<00:00,  7.62s/it]


In [23]:
print(emails[0].keys())



dict_keys(['title', 'sender', 'date_received', 'body', 'summary'])


In [14]:
display(Markdown(response.choices[0].message.content))

Okay, here’s a breakdown of your Gmail content, presented in a Markdown format, prioritizing key information:

**Key Findings & Opportunities:**

*   **Job Alerts:** You've received numerous job alerts primarily focused on AI/ML Engineering roles (Software Engineer, Machine Learning Engineer, Research Scientist) at companies like Bosch, Apple, Micron, NVIDIA, and Meta.
*   **Potential Security Alert:**  You’ve received a security alert from Google indicating that `gmail_rag_project` has access to some of your account data. You should review your Google Account activity and secure it immediately.
*   **Job Offer - Potential:** A job alert for a Senior Mechanical Engineer position at Pfizer was also detected.

**Detailed Breakdown by Email Type:**

*   **Job Alerts (LinkedIn):**  You're being actively notified about AI/ML Engineering positions across various companies, including Apple, Meta, NVIDIA, and Micron. These alerts are driving the majority of your email traffic.
*   **Security Alert (Google):** This is a critical alert indicating unauthorized access to your Google Account. You *must* investigate this immediately by checking account activity and reviewing permissions.
*   **Marketing Emails:**  You've received promotional emails from:
    *   CTOS (Financial Services) - Offering credit report services.
    *   PayLah! (Mobile Payment Platform) – Alerts about transactions and a reminder of phishing scams.
    *   Webull (Brokerage Platform) –  Promoting their services and providing links to their website, app, and social media channels.
*   **Other Emails:**
    *   A reminder about Phishing emails from Webull.
    *   A hospitalization claim email from AIA Singapore.

**Important Note:** The `gmail_rag_project` security alert is the highest priority.  Take immediate action to secure your Google Account.

Do you want me to delve deeper into any specific aspect of this information, such as:

*   Analyzing the job alerts in more detail (e.g., filtering by company or role)?
*   Providing instructions on how to investigate the security alert?

### Step 8: Create a chatbot for user to ask further questions

In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] +\
                [{"role":h["role"], "content": h["content"]} for h in history]+\
                    [{"role": "user", "content": message}]
    response = llm_client.chat.completions.create(model = llm_model, messages = messages, max_tokens = 2000)
    return response.choices[0].message.content

In [16]:
with gr.Blocks() as ui:
    with gr.Row():
        gr.ChatInterface(fn=chat)
        
ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
